# Edit State in LangSmith API 

## Steps: 
1. Have Agent.py file ready in the studio folder
2. keep .env file with LangSmith API keys and GOOGle API keys ready
3. set langgraph.json file ready
4. To start the local development server,
   run the following command in your terminal in the /studio directory in this module: BAsh
   ' langgraph dev '

### check for output:
 API: http://127.0.0.1:2024
- 🎨 Studio UI: https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024
- 📚 API Docs: http://127.0.0.1:2024/docs

This in-memory server is designed for development and testing.
For production use, please use LangSmith Deployment.

6. Copy url = http://127.0.0.1:2024

#### Note: 
- check for more info : https://docs.langchain.com/langsmith/quick-start-studio#local-development-server
- check for LangSmith setup guide in the begining of the course.

7. ### Connecting to the Local Server Using the SDK
You use the LangGraph SDK to talk to your local agent: python
```
from langgraph_sdk import get_client

url = "http://127.0.0.1:2024"
client = get_client(url=url)
```
This client object lets you:

    - list assistants
    - create threads
    - run your agent
    - stream results

8. ### Create a thread for storing event checkpoints?
A thread is a conversation session.

It stores:

    - messages
    - memory
    - state
    - checkpoints

You create one like this: python
```
thread = await client.threads.create()
```
This returns: python
```
{"thread_id": "abc123"}
```
You will use this thread ID for the entire conversation.

9.  ### Running the Agent With breakpoints
note: learn more here[API breakpoints](https://docs.langchain.com/langsmith/add-human-in-the-loop)

You run your agent like this: python
```
async for chunk in client.runs.stream(
    thread["thread_id"],
    assistant_id="agent",
    input=initial_input,
    stream_mode="values",
    interrupt_before=["llm_assistant"], # llm_node name
):
```
Note:  We can add interrupt_before=["node"] when compiling the graph that is running in Studio or with the API, you can also pass interrupt_before to the stream method directly.

#### What happens?
Your agent runs step by step:

    - user input taken
    - breakpoint to check with the user before LLm node runs for updates
    - LLM Node
    - Tool node
    - breakpoint to check with the user before LLm node runs for updates
    - LLM node
    - Final answer
After each step, LangGraph sends you a chunk containing the updated state.

## Editing/Updating the state Messages

- LangSmith API provides easy modification to State by direct access to it. We can use it as an advantage to update the state when required.
- Use await client.threads.get_state(thread['thread_id']) to acces the state
- change the ['messages']['content'] key as needed.
- easy update!

In [1]:
from langgraph_sdk import get_client

url = "http://127.0.0.1:2024"
client  = get_client(url=url)

# Search all hosted graphs
assistants = await client.assistants.search()
# list the agents
assistants

[{'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca',
  'graph_id': 'agent',
  'config': {},
  'context': {},
  'metadata': {'created_by': 'system'},
  'name': 'agent',
  'created_at': '2026-04-30T18:26:13.129954+00:00',
  'updated_at': '2026-04-30T18:26:13.129954+00:00',
  'version': 1,
  'description': None}]

In [15]:
# create a thread
thread = await client.threads.create()
print(thread)

{'thread_id': '019ddfd8-4dd6-7ab3-8e59-3b75a90a2a50', 'created_at': '2026-04-30T19:23:08.115601+00:00', 'updated_at': '2026-04-30T19:23:08.115601+00:00', 'state_updated_at': '2026-04-30T19:23:08.115601+00:00', 'metadata': {}, 'status': 'idle', 'config': {}, 'values': None}


In [16]:
# stream events
from langchain_core.messages import HumanMessage
msg_input = [ HumanMessage(content="Divide 30 and 10")]
async for event in client.runs.stream(
        thread["thread_id"],   # which conversation
        "agent",               # which assistant
        input={"messages": msg_input},       # what the user said
        stream_mode="values",  # stream full state after each step
        interrupt_before=["llm_assistant"], # add breakpoint before tool node
    ):
    #print(event)
    #capture messages
    messages = event.data.get('messages',[])
    # if messages exist
    if messages:
        print("\n", messages[-1])
    print("-" * 50)

--------------------------------------------------

 {'content': 'Divide 30 and 10', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': '8a8e2f60-8a0a-406e-85a5-9a00a8f34216'}
--------------------------------------------------


### Capture State

In [17]:
current_state = await client.threads.get_state(thread['thread_id'])
current_state

{'values': {'messages': [{'content': 'Divide 30 and 10',
    'additional_kwargs': {},
    'response_metadata': {},
    'type': 'human',
    'name': None,
    'id': '8a8e2f60-8a0a-406e-85a5-9a00a8f34216'}]},
 'next': ['llm_assistant'],
 'tasks': [{'id': '44fb0235-1b97-5301-c838-c6977ad0c413',
   'name': 'llm_assistant',
   'path': ['__pregel_pull', 'llm_assistant'],
   'error': None,
   'interrupts': [],
   'checkpoint': None,
   'state': None,
   'result': None}],
 'metadata': {'graph_id': 'agent',
  'assistant_id': 'fe096781-5601-53d2-b2f6-0d3403f7e9ca',
  'user_id': '',
  'created_by': 'system',
  'run_attempt': 1,
  'langgraph_version': '1.1.9',
  'langgraph_api_version': '0.8.1',
  'langgraph_plan': 'enterprise',
  'langgraph_host': 'self-hosted',
  'langgraph_api_url': 'http://127.0.0.1:2024',
  'thread_id': '019ddfd8-4dd6-7ab3-8e59-3b75a90a2a50',
  'run_id': '019ddfd8-507c-77c1-9b58-76a2c04fa6cc',
  'source': 'loop',
  'step': 0,
  'parents': {},
  'langgraph_auth_user_id': '',
 

last_message = current_state['values']['messages'][-1]
last_message

## get Last message

In [18]:
last_message = current_state['values']['messages'][-1]
last_message

{'content': 'Divide 30 and 10',
 'additional_kwargs': {},
 'response_metadata': {},
 'type': 'human',
 'name': None,
 'id': '8a8e2f60-8a0a-406e-85a5-9a00a8f34216'}

## Edit Last message and update the state
- agent graph also uses reducer MessagesState as the state . Updating the state appends the content 


In [19]:
last_message['content'] = "No, Multiply 100 by 10 and hten divide by 10"
last_message

{'content': 'No, Multiply 100 by 10 and hten divide by 10',
 'additional_kwargs': {},
 'response_metadata': {},
 'type': 'human',
 'name': None,
 'id': '8a8e2f60-8a0a-406e-85a5-9a00a8f34216'}

In [20]:
await client.threads.update_state(thread['thread_id'], {"messages": last_message})
current_state = await client.threads.get_state(thread['thread_id'])

In [21]:
current_state

{'values': {'messages': [{'content': 'No, Multiply 100 by 10 and hten divide by 10',
    'additional_kwargs': {},
    'response_metadata': {},
    'type': 'human',
    'name': None,
    'id': '8a8e2f60-8a0a-406e-85a5-9a00a8f34216'}]},
 'next': ['llm_assistant'],
 'tasks': [{'id': '3930da9b-1219-bf74-17a5-b5bcdca61725',
   'name': 'llm_assistant',
   'path': ['__pregel_pull', 'llm_assistant'],
   'error': None,
   'interrupts': [],
   'checkpoint': None,
   'state': None,
   'result': None}],
 'metadata': {'graph_id': 'agent',
  'thread_id': '019ddfd8-4dd6-7ab3-8e59-3b75a90a2a50',
  'source': 'update',
  'step': 1,
  'parents': {}},
 'created_at': '2026-04-30T19:23:26.951818+00:00',
 'checkpoint': {'checkpoint_id': '1f144ca1-01b8-6764-8001-df0ae1305ad1',
  'thread_id': '019ddfd8-4dd6-7ab3-8e59-3b75a90a2a50',
  'checkpoint_ns': ''},
 'parent_checkpoint': {'checkpoint_id': '1f144ca0-5d8a-6cc8-8000-7eb342e98b9c',
  'thread_id': '019ddfd8-4dd6-7ab3-8e59-3b75a90a2a50',
  'checkpoint_ns': ''}

### Run by passing None

In [22]:
async for event in client.runs.stream(
            thread["thread_id"],
            "agent",
            input=None,
            stream_mode="values",
            interrupt_before=["llm_assistant"],
        ):
            print(f"Receiving new event of type: {event.event}...")
            messages = event.data.get('messages', [])
            if messages:
                print(messages[-1])
            print("-" * 50)
        

Receiving new event of type: metadata...
--------------------------------------------------
Receiving new event of type: values...
{'content': 'No, Multiply 100 by 10 and hten divide by 10', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'human', 'name': None, 'id': '8a8e2f60-8a0a-406e-85a5-9a00a8f34216'}
--------------------------------------------------
Receiving new event of type: values...
{'content': [], 'additional_kwargs': {'function_call': {'name': 'multiply', 'arguments': '{"a": 100, "b": 10}'}, '__gemini_function_call_thought_signatures__': {'b95532c6-1dc8-4321-a23e-d7a8b97678f2': 'EjQKMgEMOdbHVxgwA6Esg1fHxR9y+XrX2ScBdCT0QwgXl1IROoNv6Te/qraoZFqmvqiFTliO'}}, 'response_metadata': {'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, 'type': 'ai', 'name': None, 'id': 'lc_run--019ddfd8-ebe2-7a62-858b-e81f1592a527-0', 'tool_calls': [{'name': 'multiply', 'args': {'a': 100, 'b': 10}, 'id': 'b95532

In [23]:
# two tool calls need llm assistant use again
async for event in client.runs.stream(
            thread["thread_id"],
            "agent",
            input=None,
            stream_mode="values",
            interrupt_before=["llm_assistant"],
        ):
            print(f"Receiving new event of type: {event.event}...")
            messages = event.data.get('messages', [])
            if messages:
                print(messages[-1])
            print("-" * 50)

Receiving new event of type: metadata...
--------------------------------------------------
Receiving new event of type: values...
{'content': '1000.0', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'tool', 'name': 'multiply', 'id': 'a1ec8078-fa0a-498a-ac00-2488caf217c2', 'tool_call_id': 'b95532c6-1dc8-4321-a23e-d7a8b97678f2', 'artifact': None, 'status': 'success'}
--------------------------------------------------
Receiving new event of type: values...
{'content': [], 'additional_kwargs': {'function_call': {'name': 'division', 'arguments': '{"a": 1000, "b": 10}'}, '__gemini_function_call_thought_signatures__': {'7b372669-282d-49b7-8648-81c00de7a590': 'EjQKMgEMOdbH6MKCI3Ow600/0FdlrATV01PRLar2NKxta7Bn/9M3ozgZHD2eSbmsOv5Klnmj'}}, 'response_metadata': {'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, 'type': 'ai', 'name': None, 'id': 'lc_run--019ddfdb-109d-7841-bec0-b304633948f7-0', 'tool_calls': [

## LLM stoped byt the breakpoint again
re run

In [24]:
#re run
async for event in client.runs.stream(
            thread["thread_id"],
            "agent",
            input=None,
            stream_mode="values",
            interrupt_before=["llm_assistant"],
        ):
            print(f"Receiving new event of type: {event.event}...")
            messages = event.data.get('messages', [])
            if messages:
                print(messages[-1])
            print("-" * 50)

Receiving new event of type: metadata...
--------------------------------------------------
Receiving new event of type: values...
{'content': '100.0', 'additional_kwargs': {}, 'response_metadata': {}, 'type': 'tool', 'name': 'division', 'id': '66c5dbb8-6d5c-4197-90e6-3b1616fa5aac', 'tool_call_id': '7b372669-282d-49b7-8648-81c00de7a590', 'artifact': None, 'status': 'success'}
--------------------------------------------------
Receiving new event of type: values...
{'content': [{'type': 'text', 'text': 'The result of multiplying 100 by 10 and then dividing by 10 is 100.', 'extras': {'signature': 'EjQKMgEMOdbHOZE1wQSn5NwhXEi5XVXY82ppdi5MzRJUtBYLVXLI/qGR6yNzfnRAMzZX6pTz'}}], 'additional_kwargs': {}, 'response_metadata': {'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, 'type': 'ai', 'name': None, 'id': 'lc_run--019ddfdd-2019-7fa2-b8a3-e49bfdb1b263-0', 'tool_calls': [], 'invalid_tool_calls': [], 'usage_metadata